# SVDynamics: reproducible dataset benchmarks

Reworked from the original `test_datasets.ipynb`. This notebook compares single-kernel SVC, fixed composite, and adaptive composite SVMs on **binary** datasets using the same stratified splits. Preprocessing and adaptive kernel weight fitting occur inside each training fold.

The original notebook included Ionosphere, Spambase, Sonar, and multiclass Dry Bean. Dry Bean is kept as a **separate multiclass baseline** because `AdaptiveCompositeKernel` currently supports binary targets only. Downloading UCI data requires internet access and `ucimlrepo` (`pip install ucimlrepo`).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from svdynamics import CompositeKernel, AdaptiveCompositeKernel, SVDClassifier

RANDOM_STATE = 42
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
SPECS = ["linear", ("rbf", {"gamma": 0.1}), ("poly", {"degree": 2, "coef0": 1})]
def models():
    return {
        "linear": SVC(kernel="linear", C=1),
        "rbf": SVC(kernel="rbf", C=1, gamma=0.1),
        "poly": SVC(kernel="poly", C=1, degree=2, coef0=1, gamma=0.1),
        "sigmoid": SVC(kernel="sigmoid", C=1),
        "fixed_composite": SVDClassifier(
            kernel=CompositeKernel(SPECS, weights=[0.2, 0.6, 0.2]), C=1),
        "adaptive_composite": SVDClassifier(
            kernel=AdaptiveCompositeKernel(SPECS), C=1),
    }
def evaluate_binary(X, y, dataset):
    X = pd.DataFrame(X).apply(pd.to_numeric, errors="coerce")
    y = LabelEncoder().fit_transform(np.asarray(y).ravel())
    if len(np.unique(y)) != 2:
        raise ValueError(f"{dataset}: adaptive composite requires binary targets")
    rows = []
    for name, estimator in models().items():
        pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), estimator)
        scores = cross_validate(pipe, X, y, cv=CV,
                                scoring={"accuracy": "accuracy", "roc_auc": "roc_auc",
                                         "ap": "average_precision"},
                                error_score="raise", n_jobs=1)
        rows.append({
            "dataset": dataset, "model": name,
            "accuracy_mean": scores["test_accuracy"].mean(),
            "accuracy_std": scores["test_accuracy"].std(),
            "roc_auc_mean": scores["test_roc_auc"].mean(),
            "ap_mean": scores["test_ap"].mean(),
        })
    return pd.DataFrame(rows)

## 1. Offline smoke test on synthetic data

Runs without network access and confirms that the evaluation code works.

In [ ]:
from sklearn.datasets import make_moons
X_moons, y_moons = make_moons(n_samples=240, noise=0.2, random_state=42)
synthetic_results = evaluate_binary(X_moons, y_moons, "Synthetic moons")
display(synthetic_results.round(3))

## 2. UCI binary datasets

These datasets require internet access. Each dataset is evaluated independently; a failed download is reported rather than silently replaced. This is a benchmark demonstration, not a tuned comparison.

In [ ]:
try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    fetch_ucirepo = None
    print("Install optional dependency: pip install ucimlrepo")

UCI_BINARY = {
    "Ionosphere": (52, "Class"),
    "Spambase": (94, "Class"),
    "Sonar": (151, "class"),
}
def load_uci_binary(dataset_id, target_column):
    dataset = fetch_ucirepo(id=dataset_id)
    X = dataset.data.features.copy()
    yframe = dataset.data.targets
    if target_column in yframe.columns:
        y = yframe[target_column]
    elif yframe.shape[1] == 1:
        y = yframe.iloc[:, 0]
    else:
        raise KeyError(f"Target {target_column} not found: {list(yframe.columns)}")
    X = X.dropna(axis=1, how="all")
    return X, y

uci_results = []
if fetch_ucirepo is not None:
    for name, (dataset_id, target) in UCI_BINARY.items():
        try:
            X_uci, y_uci = load_uci_binary(dataset_id, target)
            result = evaluate_binary(X_uci, y_uci, name)
            uci_results.append(result)
            print(f"{name}: {len(X_uci)} rows, {X_uci.shape[1]} features")
        except Exception as exc:
            print(f"{name}: could not evaluate ({type(exc).__name__}: {exc})")
if uci_results:
    binary_results = pd.concat([synthetic_results, *uci_results], ignore_index=True)
else:
    binary_results = synthetic_results.copy()
display(binary_results.round(3))

## 3. Adaptive weights learned inside training folds

This cell fits the adaptive kernel on a single training partition for interpretability. It does not use the held-out labels to learn weights.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, stratify=y_moons, test_size=0.25, random_state=42)
pipe = make_pipeline(StandardScaler(), SVDClassifier(
    kernel=AdaptiveCompositeKernel(SPECS), C=1))
pipe.fit(X_train, y_train)
fitted_kernel = pipe.steps[-1][1].kernel_
print("Learned weights:", dict(zip(["linear", "rbf", "poly"],
                                  np.round(fitted_kernel.weights_, 3))))
print("Alignments:", np.round(fitted_kernel.alignments_, 3))
print("Held-out accuracy:", round(pipe.score(X_test, y_test), 3))

## 4. Dry Bean: multiclass baseline only

The Dry Bean dataset has seven classes. The current adaptive kernel is binary-only, so compare standard SVC kernels here without trying to fit an unsupported adaptive multiclass model. Sampling to 1,000 rows limits runtime.

In [ ]:
from sklearn.model_selection import StratifiedKFold
dry_bean_results = None
if fetch_ucirepo is not None:
    try:
        bean = fetch_ucirepo(id=602)
        X_bean = bean.data.features
        y_bean = LabelEncoder().fit_transform(
            np.asarray(bean.data.targets).ravel())
        if len(y_bean) > 1000:
            from sklearn.model_selection import train_test_split
            _, X_bean, _, y_bean = train_test_split(
                X_bean, y_bean, test_size=1000 / len(y_bean),
                stratify=y_bean, random_state=42)
        rows = []
        for name, estimator in {
            "linear": SVC(kernel="linear"),
            "rbf": SVC(kernel="rbf"),
            "poly": SVC(kernel="poly"),
        }.items():
            pipe = make_pipeline(SimpleImputer(strategy="median"),
                                 StandardScaler(), estimator)
            score = cross_validate(pipe, X_bean, y_bean,
                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                scoring="accuracy", n_jobs=1, error_score="raise")
            rows.append({"model": name, "accuracy_mean": score["test_score"].mean(),
                         "accuracy_std": score["test_score"].std()})
        dry_bean_results = pd.DataFrame(rows)
        display(dry_bean_results.round(3))
    except Exception as exc:
        print(f"Dry Bean: could not evaluate ({type(exc).__name__}: {exc})")

## Interpretation and limitations

- Mean cross-validation metrics summarize performance across repeated folds; overlapping folds are not independent observations.
- Adaptive weights are learned from each training fold, not from the held-out fold.
- Kernels and hyperparameters here are illustrative, not tuned for each dataset. Do not interpret comparisons as a definitive ranking.
- For publication-quality comparisons, add nested hyperparameter tuning, multiple seeds, uncertainty intervals, and a preregistered benchmark protocol.
- Dry Bean is multiclass and is intentionally excluded from adaptive-kernel evaluation.